In [1]:
#1. regriding functions
from pathlib import Path
import numpy as np
import xarray as xr
import xesmf as xe

# ---------- Target grid ----------
target_grid_details = {
    "lon": (112, 155, 216),  # corner lon min, max, n+1
    "lat": (-45, -9, 181),   # corner lat min, max, n+1
}

def generate_grid(grid_dict):
    lons = np.linspace(*grid_dict["lon"])
    lats = np.linspace(*grid_dict["lat"])
    return lons, lats

def create_target_grid():
    lon_b, lat_b = generate_grid(target_grid_details)
    lon = 0.5 * (lon_b[:-1] + lon_b[1:])
    lat = 0.5 * (lat_b[:-1] + lat_b[1:])
    dummy = np.zeros((len(lat), len(lon)))
    ds = xr.DataArray(dummy, dims=["lat", "lon"]).to_dataset(name="target_grid")
    return ds.assign_coords({"lon": lon, "lat": lat, "lon_b": lon_b, "lat_b": lat_b})

def make_1d_bounds(coord):
    d = np.diff(coord)
    first = coord[0] - d[0] / 2
    last = coord[-1] + d[-1] / 2
    mids = 0.5 * (coord[:-1] + coord[1:])
    return np.concatenate(([first], mids, [last]))

def add_source_bounds(ds):
    lon = np.mod(ds["lon"].values, 360.0)
    lat = ds["lat"].values
    lon_b = make_1d_bounds(lon)
    lat_b = make_1d_bounds(lat)
    return ds.assign_coords(
        lon=("lon", lon),
        lon_b=("lon_b", lon_b),
        lat_b=("lat_b", lat_b),
    ).sortby("lon")

def regrid_barra2_file(in_file, out_file, var_name, method="conservative"):
    target = create_target_grid()

    with xr.open_dataset(in_file, chunks={"time": 31, "lat":646, "lon":1082}) as ds:
        ds = add_source_bounds(ds)
        ds = ds.drop_vars(["height", "level_height", "model_level_number", "sigma"], errors="ignore")


        if var_name in ds.data_vars:
            ds_use = ds[[var_name]]
        else:
            raise ValueError(f"{var_name} not found in {in_file}")

        regridder = xe.Regridder(
            ds_use, target, method=method,
            filename=str(Path(out_file).with_suffix(".weights.nc")),
            reuse_weights=False
        )

        out = regridder(ds_use, keep_attrs=True).drop_vars(
            ["lon_b", "lat_b", "time_bnds", "crs"], errors="ignore"
        )
        out.to_netcdf(out_file)

    print(f"Saved: {out_file}")




In [2]:
#2. regridding function compiled 
from pathlib import Path

def regrid_barra2_folder(
    input_dir,
    output_dir,
    var_name,
    pattern="*.nc",
    method="conservative",
    overwrite=False,
):
    input_dir = Path(input_dir)
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    files = sorted(input_dir.glob(pattern))
    print(f"Found {len(files)} files in {input_dir}")

    for i, in_file in enumerate(files, 1):
        out_file = output_dir / in_file.name.replace(".nc", "_regridded.nc")

        if out_file.exists() and not overwrite:
            print(f"[{i}/{len(files)}] Skip existing: {out_file.name}")
            continue

        try:
            print(f"[{i}/{len(files)}] Regridding: {in_file.name}")
            regrid_barra2_file(
                in_file=str(in_file),
                out_file=str(out_file),
                var_name=var_name,
                method=method,
            )
        except Exception as e:
            print(f"[{i}/{len(files)}] Failed: {in_file.name} -> {e}")


In [5]:
# 2.1 regrid hurs data 
regrid_barra2_folder(
    input_dir="/g/data/ob53/BARRA2/output/reanalysis/AUS-11/BOM/ERA5/historical/hres/BARRA-R2/v1/day/hurs/latest/",
    output_dir="/scratch/dt6/xw9650/tmp/regridded_BARRA2/",
    var_name="hurs",
    pattern="*.nc",          # or "hurs_*.nc"
    overwrite=False,
)


Found 565 files in /g/data/ob53/BARRA2/output/reanalysis/AUS-11/BOM/ERA5/historical/hres/BARRA-R2/v1/day/hurs/latest
[1/565] Skip existing: hurs_AUS-11_ERA5_historical_hres_BOM_BARRA-R2_v1_day_197901-197901_regridded.nc
[2/565] Skip existing: hurs_AUS-11_ERA5_historical_hres_BOM_BARRA-R2_v1_day_197902-197902_regridded.nc
[3/565] Skip existing: hurs_AUS-11_ERA5_historical_hres_BOM_BARRA-R2_v1_day_197903-197903_regridded.nc
[4/565] Skip existing: hurs_AUS-11_ERA5_historical_hres_BOM_BARRA-R2_v1_day_197904-197904_regridded.nc
[5/565] Skip existing: hurs_AUS-11_ERA5_historical_hres_BOM_BARRA-R2_v1_day_197905-197905_regridded.nc
[6/565] Skip existing: hurs_AUS-11_ERA5_historical_hres_BOM_BARRA-R2_v1_day_197906-197906_regridded.nc
[7/565] Skip existing: hurs_AUS-11_ERA5_historical_hres_BOM_BARRA-R2_v1_day_197907-197907_regridded.nc
[8/565] Skip existing: hurs_AUS-11_ERA5_historical_hres_BOM_BARRA-R2_v1_day_197908-197908_regridded.nc
[9/565] Skip existing: hurs_AUS-11_ERA5_historical_hres_BOM

In [4]:
#2.2 regrid tas data
regrid_barra2_folder(
    input_dir="/g/data/ob53/BARRA2/output/reanalysis/AUS-11/BOM/ERA5/historical/hres/BARRA-R2/v1/day/tas/latest/",
    output_dir="/scratch/dt6/xw9650/tmp/regridded_BARRA2/",
    var_name="tas",
    pattern="*.nc",          # or "hurs_*.nc"
    overwrite=False,
)


Found 565 files in /g/data/ob53/BARRA2/output/reanalysis/AUS-11/BOM/ERA5/historical/hres/BARRA-R2/v1/day/tas/latest
[1/565] Skip existing: tas_AUS-11_ERA5_historical_hres_BOM_BARRA-R2_v1_day_197901-197901_regridded.nc
[2/565] Skip existing: tas_AUS-11_ERA5_historical_hres_BOM_BARRA-R2_v1_day_197902-197902_regridded.nc
[3/565] Skip existing: tas_AUS-11_ERA5_historical_hres_BOM_BARRA-R2_v1_day_197903-197903_regridded.nc
[4/565] Skip existing: tas_AUS-11_ERA5_historical_hres_BOM_BARRA-R2_v1_day_197904-197904_regridded.nc
[5/565] Skip existing: tas_AUS-11_ERA5_historical_hres_BOM_BARRA-R2_v1_day_197905-197905_regridded.nc
[6/565] Skip existing: tas_AUS-11_ERA5_historical_hres_BOM_BARRA-R2_v1_day_197906-197906_regridded.nc
[7/565] Skip existing: tas_AUS-11_ERA5_historical_hres_BOM_BARRA-R2_v1_day_197907-197907_regridded.nc
[8/565] Skip existing: tas_AUS-11_ERA5_historical_hres_BOM_BARRA-R2_v1_day_197908-197908_regridded.nc
[9/565] Skip existing: tas_AUS-11_ERA5_historical_hres_BOM_BARRA-R2_